## Setup


In [ ]:
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

SCRIPT = "matmul_multi_device_split.py"
N = 8192
ALPHAS = np.round(np.arange(0.0, 1.01, 0.1), 2)
RUNS = 3
WARMUP = 1


## Helper


In [ ]:
GLOBAL_GFLOPS_RE = r"Global GFLOPS:\s+([0-9.+-eE]+)"
TOTAL_TIME_RE = r"Total time.*:\s+([0-9.+-eE]+)"
MEAN_TIME_RE = r"Mean time per run.*:\s+([0-9.+-eE]+)"
SPEEDUP_RE = r"Speedup vs NVIDIA-only:\s+x([0-9.+-eE]+)"

def _extract_metric(pattern, text, label, required=True):
    m = re.search(pattern, text)
    if m is None:
        if required:
            raise RuntimeError(f"Could not parse {label}.")
        return None
    return float(m.group(1))

def _extract_t_mean(text):
    mean_time = _extract_metric(MEAN_TIME_RE, text, "Mean time per run", required=False)
    if mean_time is not None:
        return mean_time
    total_time = _extract_metric(TOTAL_TIME_RE, text, "Total time", required=True)
    return total_time / RUNS

def run_alpha(alpha):
    cmd = [
        "python", SCRIPT,
        "--n", str(N),
        "--alpha", str(float(alpha)),
        "--runs", str(RUNS),
        "--warmup", str(WARMUP)
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    out = result.stdout
    err = result.stderr

    if result.returncode != 0:
        raise RuntimeError(f"Command failed for alpha={alpha}:\nSTDOUT:\n{out}\nSTDERR:\n{err}")

    t_mean = _extract_t_mean(out)
    gflops = (2.0 * (N ** 3)) / (t_mean * 1e9)
    speedup = _extract_metric(SPEEDUP_RE, out, "Speedup", required=False)

    return {
        "alpha": float(alpha),
        "gflops": gflops,
        "time": t_mean,
        "speedup": speedup
    }


## Sweep


In [ ]:
results = []

for alpha in ALPHAS:
    print(f"Running alpha={alpha:.2f}")
    results.append(run_alpha(alpha))

df = pd.DataFrame(results).sort_values("alpha").reset_index(drop=True)
df


## Save


In [ ]:
df.to_csv("alpha_sweep_results.csv", index=False)


## Plots


In [ ]:
plt.figure()
plt.plot(df["alpha"], df["gflops"], marker="o")
plt.xlabel("alpha (fraction on NVIDIA)")
plt.ylabel("GFLOPS")
plt.title("Multi-device performance vs alpha")
plt.grid()
plt.show()


In [ ]:
plt.figure()
plt.plot(df["alpha"], df["speedup"], marker="o")
plt.xlabel("alpha (fraction on NVIDIA)")
plt.ylabel("Speedup vs NVIDIA uncoalesced")
plt.title("Speedup vs alpha")
plt.grid()
plt.show()


In [ ]:
plt.figure()
plt.plot(df["alpha"], df["time"], marker="o")
plt.xlabel("alpha")
plt.ylabel("Mean time per run (s)")
plt.title("Execution time vs alpha")
plt.grid()
plt.show()


## Best Alpha


In [ ]:
best = df.loc[df["gflops"].idxmax()]
best
